In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

DataFrame[]

In [0]:
volume = "/Volumes/workspace/raw/bronze"

tabela_variavel = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

for arquivo, tabela in tabela_variavel.items():
    df = spark.read.csv(f"{volume}/{arquivo}", header=True, inferSchema=True)
    df = df.withColumn("ingestion_datetime", F.current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(f"workspace.bronze.{tabela}")

In [0]:
from datetime import date, timedelta

data_fim_default = date.today()
data_inicio_default = data_fim_default - timedelta(days=7)

dbutils.widgets.text("data_inicio", data_inicio_default.strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", data_fim_default.strftime("%m-%d-%Y"))

In [0]:
import requests

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

resp = requests.get(url)
dados = resp.json()["value"]

if not dados:
    raise ValueError(
        f"API não retornou cotações para o intervalo. Confira se os widgets estão preenchidos e se o formato é MM-DD-AAAA."
    )

df_cotacao = spark.createDataFrame([Row(**d) for d in dados])


df_cotacao = df_cotacao.withColumn("ingestion_datetime",current_timestamp())

df_cotacao.write.format("delta").mode("append").saveAsTable("workspace.bronze.tb_cotacao_dolar")

display(df_cotacao)

print("URL:", url)
print("Status:", resp.status_code)
print("Resposta:", resp.text[:500])



cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.169,2026-09-14 13:10:08.144425,2026-09-19T23:20:19.098Z
5.1484,2026-09-15 13:09:19.199664,2026-09-19T23:20:19.098Z
5.152,2026-09-16 13:05:30.35873,2026-09-19T23:20:19.098Z
5.1515,2026-09-17 13:03:21.858212,2026-09-19T23:20:19.098Z
5.1569,2026-09-18 13:03:34.742036,2026-09-19T23:20:19.098Z


URL: https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-13-2026'&@dataFinalCotacao='09-19-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json
Status: 200
Resposta: {"@odata.context":"https://was-p.bcnet.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata$metadata#_CotacaoDolarPeriodo(cotacaoCompra,dataHoraCotacao)","value":[{"cotacaoCompra":5.16900,"dataHoraCotacao":"2026-09-14 13:10:08.144425"},{"cotacaoCompra":5.14840,"dataHoraCotacao":"2026-09-15 13:09:19.199664"},{"cotacaoCompra":5.15200,"dataHoraCotacao":"2026-09-16 13:05:30.35873"},{"cotacaoCompra":5.15150,"dataHoraCotacao":"2026-09-17 13:03:21.858212"},{"cotacaoCompra":5.15690,"dataHoraCotacao":"2026-09-
